# Lecture 6a: THE PROVIDER'S SIDE - Building the Authenticated Structure

> **DOMAIN BANNER - read this first.** In this notebook YOU ARE THE
> PROVIDER. There is exactly **one** of you per log. You hold the
> signing key. You own a Lean toolchain and hours of compute. You
> carry the append-only obligations. Nothing in this notebook is
> ever executed by an agent - and that asymmetry is not an
> implementation detail, it is the entire design (see the
> justification at the end).

The provider's job, end to end: **verify -> leaf -> tree -> sign ->
self-check**. Only the first step involves Lean; everything after
is hashing and one signature.


## Learning Objectives

- Build the full authenticated data structure from real attestations: leaves, tree, Signed Tree Head.
- Place the Lean verification correctly: it is the LEAF-MAKING step, the only expensive one, and it never travels to the agent.
- Sign the root with the merkleized library and run the provider's own inclusion self-check ("the provider eats its own dogfood").
- Justify the singleton/many split as a design decision.


## Step 1 - Verify (the expensive step, done ONCE)

The leaf content is a signed **attestation**: the outcome of replaying
every Lean proof of one repository under lean-guard (~30 minutes of
kernel re-checking per fork on the reference machine). This notebook
does NOT re-run that - the shipped `evidence/` attestations ARE that
step's output. What matters architecturally: **the Lean cost lives
here and only here.** No agent ever pays it again.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "provider" / "src"))

from pacta.yamlio import load_data

attestations = {
    fork: load_data(repo_root / "evidence" / f"{fork}-ed25519.attestation.yaml")
    for fork in ["dalek", "anza", "risc0", "betrusted"]
}
for fork, att in attestations.items():
    certs = att["certificates"]
    clean = sum(1 for c in certs if c["status"] == "proven" and c["axiom_status"] == "clean")
    print(f"{fork}: {clean}/{len(certs)} proven | commit {att['subject']['repo_commit'][:8]} | guard: {att['machine_protection']['lean_guard'].rsplit('/',1)[-1]}")


## Steps 2+3 - Leaf and tree (cheap, mechanical)

Each attestation is wrapped, canonically serialized, and hashed with
the RFC 9162 leaf prefix `0x00`; pairs of nodes hash with prefix
`0x01`. Build a REAL provider log in a scratch directory - you are
the provider, so mint your own key first:


In [ ]:
import tempfile
from pacta.signing import generate_ed25519_keypair
from pacta_provider.transparency_log import TransparencyLog

state = Path(tempfile.mkdtemp(prefix="provider-lecture-"))
generate_ed25519_keypair(state / "provider.key", state / "provider.pub")
log = TransparencyLog(state / "log")
log.init("lecture-provider", state / "provider.pub")

receipts = {}
for fork, att in attestations.items():
    att_path = state / f"{fork}.attestation.yaml"
    from pacta.yamlio import dump_data
    dump_data(att, att_path)
    receipts[fork] = log.append_attestation(att_path, state / "provider.key", state / "provider.pub")
print("tree size:", receipts["betrusted"]["tree_size"])
print("root:", receipts["betrusted"]["sth"]["root_hash"][:32], "…")


## Steps 4+5 - Sign the root, then CHECK YOURSELF

The tree head is signed with the **merkleized library itself** (when
the dogfood binary is built): the Ed25519 code that signs this root
is the same pinned dalek source whose proof attestation is a leaf of
this very tree. Before signing, the provider runs the SAME Merkle
inclusion verification an agent would run - on its own signing
library's leaf, against the tree it is about to sign - and embeds
the verdict in the signature block. A root signature that names the
leaf vouching for the code that produced it:


In [ ]:
import json

sth = receipts["betrusted"]["sth"]
ed = sth["signatures"]["ed25519"]
print("signing backend:", ed.get("signing_backend"))
print(json.dumps(ed.get("signing_provenance", {"note": "dogfood binary not built on this host - OpenSSL fallback, provenance omitted"}), indent=1))


## The structure you just built, drawn from your own log


In [ ]:
from pacta.transparency import leaf_hash, merkle_root, node_hash

def svg_merkle(leaf_hashes, highlight=None, title="", domain="PROVIDER: builds every box below", color="#1e7f4f"):
    n = len(leaf_hashes)
    width, lh, lv = 980, 108, 92
    levels = []
    level = [bytes.fromhex(h) if isinstance(h, str) else h for h in leaf_hashes]
    levels.append(level)
    while len(level) > 1:
        nxt = []
        for i in range(0, len(level) - 1, 2):
            nxt.append(node_hash(level[i], level[i + 1]))
        if len(level) % 2:
            nxt.append(level[-1])
        levels.append(nxt)
        level = nxt
    height = 130 + lv * len(levels) + 60
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" font-family="monospace" font-size="11">']
    out.append(f'<rect x="4" y="4" width="{width-8}" height="{height-8}" fill="none" stroke="{color}" stroke-width="2" rx="8"/>')
    out.append(f'<text x="16" y="24" fill="{color}" font-size="13" font-weight="bold">{domain}</text>')
    out.append(f'<text x="16" y="42" fill="#555">{title}</text>')
    pos = {}
    for li, lvl in enumerate(levels):
        y = height - 70 - li * lv
        span = width / (len(lvl) + 1)
        for i, node in enumerate(lvl):
            x = span * (i + 1)
            pos[(li, i)] = (x, y)
            hl = highlight and li == 0 and i == highlight[0]
            sib = highlight and (li, i) in highlight[1]
            fill = "#fdf0da" if sib else ("#e2f2e9" if hl else "#f4f4f6")
            stroke = "#a86a10" if sib else ("#1e7f4f" if hl else "#999")
            out.append(f'<rect x="{x-52}" y="{y-16}" width="104" height="32" fill="{fill}" stroke="{stroke}" stroke-width="{2 if (hl or sib) else 1}" rx="4"/>')
            label = ("leaf %d" % i) if li == 0 else ("root" if li == len(levels) - 1 else "node")
            out.append(f'<text x="{x}" y="{y-3}" text-anchor="middle" fill="#333">{label}</text>')
            out.append(f'<text x="{x}" y="{y+11}" text-anchor="middle" fill="#777">{node.hex()[:10]}…</text>')
            if li > 0:
                for ci in (2 * i, 2 * i + 1):
                    if (li - 1, ci) in pos:
                        cx, cy = pos[(li - 1, ci)]
                        out.append(f'<line x1="{x}" y1="{y+16}" x2="{cx}" y2="{cy-16}" stroke="#bbb"/>')
    rx, ry = pos[(len(levels) - 1, 0)]
    out.append(f'<rect x="{rx-135}" y="{ry-62}" width="270" height="30" fill="#eef" stroke="#3b4d8f" rx="4"/>')
    out.append(f'<text x="{rx}" y="{ry-42}" text-anchor="middle" fill="#3b4d8f">Signed Tree Head: Ed25519(root) via merkleized library</text>')
    out.append(f'<line x1="{rx}" y1="{ry-32}" x2="{rx}" y2="{ry-16}" stroke="#3b4d8f"/>')
    out.append("</svg>")
    return "".join(out)

entries = log.entries()
leaf_hexes = [leaf_hash(e.leaf_bytes()).hex() for e in entries]
svg = svg_merkle(leaf_hexes, title=f"your lecture log: {len(entries)} attestation leaves, root {merkle_root([e.leaf_bytes() for e in entries]).hex()[:16]}…")
try:
    from IPython.display import SVG, display
    display(SVG(svg))
except Exception:
    print(svg[:200], "… (open in a notebook to render)")


## Why a singleton? The design justification

| | Provider (this notebook) | Agent (next notebook) |
|---|---|---|
| How many | **exactly one** per log | unbounded |
| Owns | the signing key, the Lean toolchain, the full log | the provider's PUBLIC key, a pin file |
| Pays | hours of kernel time per repo, ONCE | milliseconds per check, forever |
| Obligations | append-only, sign every head, serve proofs, self-verify | pin every head, demand consistency |
| Can be wrong? | detectably: signatures + pins make lies attributable | fails closed |

The asymmetry is the product. If agents had to run Lean, the service
would add nothing; if the provider's claims weren't pinned and
signed, trust would be a rumor. Every artifact in this course lives
on exactly one side of this table - and the split between this
notebook and the next MIRRORS it on purpose: if you cannot say
which notebook a step belongs to, you have not understood the step.

## Exercises

- Append a fifth attestation (edit one field of a copy) and watch the root change; which internal nodes changed and which did not? Explain from the tree shape.
- The self-inclusion check ran against the tree BEFORE your key existed in any leaf. What does `signing_provenance.self_inclusion` say, and why is recording that honest?
- Cost accounting: with 4 repos x 30 minutes of Lean and N agents x 5 ms of verification, at what N does the provider model beat every-agent-verifies-locally? (Hint: N=1.)
- Design question: what breaks if there are TWO providers with one key? With two keys and one log?
